In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

import joblib

In [2]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import os

file_path = None

for root, dirs, files in os.walk('/content/drive/MyDrive'):
    if 'DATA.xlsx' in files:
        file_path = os.path.join(root, 'DATA.xlsx')
        break

print("Dataset path:", file_path)

Dataset path: /content/drive/MyDrive/Cell-free DNA as an Independent Predictor of Sever/Cell-free DNA as an Independent Predictor of Sever/DATA.xlsx


In [4]:
import pandas as pd

df = pd.read_excel(file_path)

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (116, 32)

Columns:
['number', 'GROUP', 'GROUP2', 'sex', 'age', 'weight', 'height', 'BMI', 'Cardiovascular_disease', 'hypertension', 'diabetes', 'Cerebrovascular_disease', 'Hyperlipidemia', 'smoking', 'alcohol', 'PLR', 'NLR', 'CRP', 'PCT', 'IL2', 'IL4', 'IL6', 'IL10', 'TNFα', 'IFNγ', 'IL17A', 'IL1β', 'IL5', 'IL12p70', 'IFNα', 'IL8', 'cfDNA']


In [5]:
df.head()

,number,GROUP,GROUP2,sex,age,weight,height,BMI,Cardiovascular_disease,hypertension,...,IL10,TNFα,IFNγ,IL17A,IL1β,IL5,IL12p70,IFNα,IL8,cfDNA
0,1,NSCAP,CAP,0,68,62.0,163.0,23.335466,0,1,...,4.3,0.7,2.69,2.4,0.4,0.2,0.3,2.6,12.72,4.700000
1,1,NSCAP,CAP,1,68,66.0,174.0,21.799445,0,0,...,4.9,1.3,2.90,0.8,0.4,0.1,0.2,1.6,7.10,5.900000
2,1,NSCAP,CAP,0,61,57.0,155.0,23.725286,1,1,...,2.1,0.1,1.00,1.1,0.4,0.9,0.4,0.4,9.20,7.515773
3,1,NSCAP,CAP,0,70,56.0,158.0,22.432303,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,8.400000
4,1,NSCAP,CAP,0,95,62.0,163.0,23.335466,1,0,...,2.1,0.7,0.30,0.7,0.9,1.4,0.9,0.1,19.90,8.600000


In [8]:
# Keep only CRP and CAP severity information

crp_df = df[["CRP", "GROUP"]].copy()

print(crp_df.head())
print("\nShape:", crp_df.shape)

      CRP  GROUP
0    6.97  NSCAP
1    3.87  NSCAP
2    1.74  NSCAP
3    5.94  NSCAP
4  110.31  NSCAP

Shape: (116, 2)


In [9]:
print(crp_df["GROUP"].value_counts())

GROUP
Control    42
NSCAP      40
SCAP       34
Name: count, dtype: int64


In [10]:
crp_df["Target"] = crp_df["GROUP"].map({
    "NSCAP": 0,
    "SCAP": 1
})

print(crp_df.head())
print("\nTarget distribution:")
print(crp_df["Target"].value_counts())

      CRP  GROUP  Target
0    6.97  NSCAP     0.0
1    3.87  NSCAP     0.0
2    1.74  NSCAP     0.0
3    5.94  NSCAP     0.0
4  110.31  NSCAP     0.0

Target distribution:
Target
0.0    40
1.0    34
Name: count, dtype: int64


In [11]:
X = crp_df[["CRP"]].values
y = crp_df["Target"].values

print("X shape:", X.shape)
print("y shape:", y.shape)

print("\nFirst 5 X values:")
print(X[:5])

print("\nFirst 5 y values:")
print(y[:5])

X shape: (116, 1)
y shape: (116,)

First 5 X values:
[[  6.97]
 [  3.87]
 [  1.74]
 [  5.94]
 [110.31]]

First 5 y values:
[0. 0. 0. 0. 0.]


In [13]:
from sklearn.model_selection import train_test_split
import numpy as np

# Create a boolean mask to identify non-NaN values in y
not_nan_mask = ~np.isnan(y)

# Filter X and y to remove NaN values
X_filtered = X[not_nan_mask]
y_filtered = y[not_nan_mask]

X_train, X_test, y_train, y_test = train_test_split(
    X_filtered, # Use the filtered X
    y_filtered, # Use the filtered y
    test_size=0.20,
    random_state=42,
    stratify=y_filtered # Stratify by the filtered y
)

print("X_train shape:", X_train.shape)
print("X_test shape:", X_test.shape)
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

X_train shape: (59, 1)
X_test shape: (15, 1)
y_train shape: (59,)
y_test shape: (15,)


In [14]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

print("Scaled X_train:")
print(X_train[:5])

Scaled X_train:
[[ 2.26129571]
 [ 1.43950751]
 [-0.94634283]
 [ 1.11447188]
 [-1.10198774]]


In [15]:
import tensorflow as tf

model = tf.keras.Sequential([
    tf.keras.layers.Input(shape=(1,)),
    tf.keras.layers.Dense(16, activation="relu"),
    tf.keras.layers.Dense(8, activation="relu"),
    tf.keras.layers.Dense(1, activation="sigmoid")
])

model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense (Dense)                   │ (None, 16)             │            32 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 8)              │           136 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │             9 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 177 (708.00 B)

 Trainable params: 177 (708.00 B)

 Non-trainable params: 0 (0.00 B)

In [16]:
history = model.fit(
    X_train,
    y_train,
    epochs=50,
    batch_size=16,
    validation_split=0.2,
    verbose=1
)

Epoch 1/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 116ms/step - accuracy: 0.4043 - loss: 0.7031 - val_accuracy: 0.5833 - val_loss: 0.6997
Epoch 2/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 38ms/step - accuracy: 0.4043 - loss: 0.6999 - val_accuracy: 0.5000 - val_loss: 0.6957
Epoch 3/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.4043 - loss: 0.6969 - val_accuracy: 0.5000 - val_loss: 0.6920
Epoch 4/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step - accuracy: 0.3830 - loss: 0.6937 - val_accuracy: 0.7500 - val_loss: 0.6885
Epoch 5/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 34ms/step - accuracy: 0.5532 - loss: 0.6909 - val_accuracy: 0.7500 - val_loss: 0.6851
Epoch 6/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 58ms/step - accuracy: 0.5532 - loss: 0.6888 - val_accuracy: 0.7500 - val_loss: 0.6813
Epoch 7/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 53ms/step - accuracy: 0.5745 - loss: 0.6857 - val_accuracy: 0.7500 - val_loss: 0.6780
Epoch 8/50
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 68ms/step - accuracy: 0.5745 - loss: 0.6833 - val_accuracy: 0.7500 - val_loss: 0.6748

In [17]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

y_prob = model.predict(X_test).ravel()
y_pred = (y_prob >= 0.5).astype(int)

print("Accuracy:", accuracy_score(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

print("\nClassification Report:")
print(classification_report(
    y_test,
    y_pred,
    target_names=["NSCAP", "SCAP"]
))

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 203ms/step
Accuracy: 0.6

Confusion Matrix:
[[6 2]
 [4 3]]

Classification Report:
              precision    recall  f1-score   support

       NSCAP       0.60      0.75      0.67         8
        SCAP       0.60      0.43      0.50         7

    accuracy                           0.60        15
   macro avg       0.60      0.59      0.58        15
weighted avg       0.60      0.60      0.59        15



In [18]:
print(crp_df["GROUP"].value_counts())

GROUP
Control    42
NSCAP      40
SCAP       34
Name: count, dtype: int64


In [19]:
model.save("/content/drive/MyDrive/crp_model.keras")

print("CRP model saved successfully!")

CRP model saved successfully!


In [20]:
import joblib

joblib.dump(
    scaler,
    "/content/drive/MyDrive/crp_scaler.pkl"
)

print("CRP scaler saved successfully!")

CRP scaler saved successfully!
